In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn import preprocessing, metrics
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn import svm

from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.model_selection import KFold

In [ ]:
df = pd.read_excel("bbd final.xlsx")
df.head()

,Delay,SFM,Corr de pearson,Variance,Kurtosis,Skewness,variable cible
0,-0.28,0.016455,0.694238,29907.732710,0.242694,-0.092738,0
1,-0.04,0.013650,0.832612,13435.613358,7.351518,-1.066323,0
2,-0.04,0.065356,0.714471,29691.588877,2.971953,-1.010291,0
3,-0.08,0.012884,0.703407,63469.498183,2.946650,-0.771221,0
4,-0.12,0.014145,0.755860,46445.023357,8.160552,1.067372,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Delay            65 non-null     float64
 1   SFM              65 non-null     float64
 2   Corr de pearson  65 non-null     float64
 3   Variance         65 non-null     float64
 4   Kurtosis         65 non-null     float64
 5   Skewness         65 non-null     float64
 6   variable cible   65 non-null     int64  
dtypes: float64(6), int64(1)
memory usage: 3.7 KB


In [ ]:
df.describe()

,Delay,SFM,Corr de pearson,Variance,Kurtosis,Skewness,variable cible
count,65.000000,65.000000,65.000000,6.500000e+01,65.000000,65.000000,65.000000
mean,-0.809846,0.052858,0.676752,9.703690e+04,15.194270,-0.193020,0.538462
std,3.233744,0.050903,0.185414,2.035666e+05,27.645958,2.499131,0.502398
min,-23.880000,0.001765,0.146754,3.622223e+03,-1.264217,-6.209980,0.000000
25%,-0.160000,0.015493,0.588323,1.989526e+04,1.186361,-1.574471,0.000000
50%,-0.040000,0.027135,0.714471,3.079102e+04,6.197141,-0.333756,1.000000
75%,0.040000,0.086302,0.805542,8.069674e+04,13.384323,0.461248,1.000000
max,0.200000,0.191375,0.944920,1.340351e+06,153.821435,7.159344,1.000000


In [ ]:
X = df.drop(columns=["variable cible"])
y = df["variable cible"]

X_array = X.values
y_array = y.values

#  KFold_train_test_split

In [ ]:
def kfold_train_test_split(X, y, k):
    kf = KFold(n_splits=k, shuffle=True, random_state=4)

    # Stocker les données de chaque itération
    X_train_list, X_test_list, y_train_list, y_test_list = [], [], [], []

    for train_index, test_index in kf.split(X, y):
        print(train_index, test_index)
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        # Sur-échantillonnage de la classe minoritaire
        X_minority = X_train[y_train == 1]
        y_minority = y_train[y_train == 1]
        X_minority_resampled, y_minority_resampled = resample(X_minority,y_minority, replace=True,n_samples=X_train[y_train == 0].shape[0],
        random_state=42)

        # Concaténation des données sur-échantillonnées avec les données originales
        X_train_balanced = np.vstack((X_train[y_train == 0], X_minority_resampled))
        y_train_balanced = np.hstack((y_train[y_train == 0], y_minority_resampled))

        print(type(X_train))
        X_train_list.append(X_train_balanced)
        X_test_list.append(X_test)
        y_train_list.append(y_train_balanced)
        y_test_list.append(y_test)
        print("end_of_iteration")
    return X_train_list, X_test_list, y_train_list, y_test_list


In [ ]:
X_train_list, X_test_list, y_train_list, y_test_list = kfold_train_test_split(X_array, y_array, 3)

[ 0  1  2  3  5  8  9 13 15 17 18 19 21 23 27 28 30 32 33 34 36 38 39 40
 41 42 44 45 46 48 49 50 51 52 53 55 56 57 58 59 60 63 64] [ 4  6  7 10 11 12 14 16 20 22 24 25 26 29 31 35 37 43 47 54 61 62]
<class 'numpy.ndarray'>
end_of_iteration
[ 0  1  3  4  5  6  7  8  9 10 11 12 14 16 20 21 22 23 24 25 26 29 30 31
 35 36 37 38 39 40 42 43 44 45 46 47 50 54 55 57 60 61 62] [ 2 13 15 17 18 19 27 28 32 33 34 41 48 49 51 52 53 56 58 59 63 64]
<class 'numpy.ndarray'>
end_of_iteration
[ 2  4  6  7 10 11 12 13 14 15 16 17 18 19 20 22 24 25 26 27 28 29 31 32
 33 34 35 37 41 43 47 48 49 51 52 53 54 56 58 59 61 62 63 64] [ 0  1  3  5  8  9 21 23 30 36 38 39 40 42 44 45 46 50 55 57 60]
<class 'numpy.ndarray'>
end_of_iteration


In [ ]:
# La première itération
X_train_fold1 = X_train_list[0]
X_test_fold1 = X_test_list[0]
y_train_fold1 = y_train_list[0]
y_test_fold1 = y_test_list[0]

In [ ]:
len(X_train_list)

3

In [ ]:
# Afficher les shapes pour chaque itération
for i in range(len(X_train_list)):
    X_train_shape = X_train_list[i].shape
    X_test_shape = X_test_list[i].shape
    y_train_shape = y_train_list[i].shape
    y_test_shape = y_test_list[i].shape

    print(f"Iteration {i+1}:")
    print(f"X_train shape: {X_train_shape}")
    print(f"X_test shape: {X_test_shape}")
    print(f"y_train shape: {y_train_shape}")
    print(f"y_test shape: {y_test_shape}")
    print()


Iteration 1:
X_train shape: (43, 6)
X_test shape: (22, 6)
y_train shape: (43,)
y_test shape: (22,)

Iteration 2:
X_train shape: (43, 6)
X_test shape: (22, 6)
y_train shape: (43,)
y_test shape: (22,)

Iteration 3:
X_train shape: (44, 6)
X_test shape: (21, 6)
y_train shape: (44,)
y_test shape: (21,)



In [ ]:
param_grid = {
    'KNeighborsClassifier': {'n_neighbors': [3, 5, 7]},
    'DecisionTreeClassifier': {'max_depth': [3, 5, 7]},
    'SVC': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']},
    'RandomForestClassifier': {'n_estimators': [50, 100, 200]},
    'LogisticRegression': {'C': [0.1, 1, 10], 'solver': ['lbfgs'], 'max_iter': [1000]}
}

In [ ]:
classifiers = [
    GridSearchCV(KNeighborsClassifier(), param_grid['KNeighborsClassifier'], cv=5),
    GridSearchCV(DecisionTreeClassifier(), param_grid['DecisionTreeClassifier'], cv=5),
    GridSearchCV(SVC(), param_grid['SVC'], cv=5),
    GridSearchCV(RandomForestClassifier(), param_grid['RandomForestClassifier'], cv=5),
    GridSearchCV(LogisticRegression(), param_grid['LogisticRegression'], cv=5)
]

In [ ]:
for grid_search in classifiers:
    classifier_name = grid_search.estimator.__class__.__name__
    print(f"\nClassifier: {classifier_name}")

    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    specificity_scores = []
    f1_scores = []

    for i in range(len(X_train_list)):

        X_train = X_train_list[i]
        X_test = X_test_list[i]
        y_train = y_train_list[i]
        y_test = y_test_list[i]

        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Entraîner le modèle avec GridSearchCV
        grid_search.fit(X_train_scaled, y_train)

        # Obtenir les prédictions sur les données de test avec le meilleur modèle trouvé par GridSearchCV
        y_pred = grid_search.predict(X_test_scaled)

        # Calcul des scores pour chaque fold
        accuracy_fold = accuracy_score(y_test, y_pred)
        precision_fold = precision_score(y_test, y_pred)
        recall_fold = recall_score(y_test, y_pred)

        # Calcul de la spécificité
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        specificity_fold = tn / (tn + fp) if (tn + fp) != 0 else 0

        f1_fold = f1_score(y_test, y_pred)

        print(f"Iteration {i+1} - Accuracy: {accuracy_fold:.4f}, Precision: {precision_fold:.4f}, Recall: {recall_fold:.4f}, Specificity: {specificity_fold:.4f}, F1-Score: {f1_fold:.4f}")

        accuracy_scores.append(accuracy_fold)
        precision_scores.append(precision_fold)
        recall_scores.append(recall_fold)
        specificity_scores.append(specificity_fold)
        f1_scores.append(f1_fold)

    # Calcul des scores moyens sur les données de test
    average_accuracy = np.mean(accuracy_scores)
    average_precision = np.mean(precision_scores)
    average_recall = np.mean(recall_scores)
    average_specificity = np.mean(specificity_scores)
    average_f1 = np.mean(f1_scores)

    print()
    print(f"Average Accuracy : {average_accuracy * 100:.2f} %")
    print(f"Average Precision : {average_precision * 100:.2f} %")
    print(f"Average Recall across : {average_recall * 100:.2f} %")
    print(f"Average Specificity across : {average_specificity * 100:.2f} %")
    print(f"Average F1-Score across : {average_f1 * 100:.2f} %")

    print()
    print(classification_report(y_test, y_pred))



Classifier: KNeighborsClassifier
Iteration 1 - Accuracy: 0.5455, Precision: 0.4286, Recall: 0.7500, Specificity: 0.4286, F1-Score: 0.5455
Iteration 2 - Accuracy: 0.5909, Precision: 0.7273, Recall: 0.5714, Specificity: 0.6250, F1-Score: 0.6400
Iteration 3 - Accuracy: 0.4762, Precision: 0.5714, Recall: 0.6154, Specificity: 0.2500, F1-Score: 0.5926

Average Accuracy : 53.75 %
Average Precision : 57.58 %
Average Recall across : 64.56 %
Average Specificity across : 43.45 %
Average F1-Score across : 59.27 %

              precision    recall  f1-score   support

           0       0.29      0.25      0.27         8
           1       0.57      0.62      0.59        13

    accuracy                           0.48        21
   macro avg       0.43      0.43      0.43        21
weighted avg       0.46      0.48      0.47        21


Classifier: DecisionTreeClassifier
Iteration 1 - Accuracy: 0.5455, Precision: 0.4167, Recall: 0.6250, Specificity: 0.5000, F1-Score: 0.5000
Iteration 2 - Accuracy: